In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 5


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.6262583881616592
Epoch 2/100, Loss: 1.7118563428521156
Epoch 3/100, Loss: 1.6691074073314667
Epoch 4/100, Loss: 1.7657701894640923
Epoch 5/100, Loss: 1.7789205946028233
Epoch 6/100, Loss: 1.614033356308937
Epoch 7/100, Loss: 1.6438283994793892
Epoch 8/100, Loss: 1.6657912135124207
Epoch 9/100, Loss: 1.5589835792779922
Epoch 10/100, Loss: 1.6175701692700386
Epoch 11/100, Loss: 1.6457806900143623
Epoch 12/100, Loss: 1.6617464870214462
Epoch 13/100, Loss: 1.6706591919064522
Epoch 14/100, Loss: 1.7715276703238487
Epoch 15/100, Loss: 1.693850614130497
Epoch 16/100, Loss: 1.622896820306778


Epoch 17/100, Loss: 1.7082300707697868
Epoch 18/100, Loss: 1.6378674507141113
Epoch 19/100, Loss: 1.676350198686123
Epoch 20/100, Loss: 1.6708998084068298
Epoch 21/100, Loss: 1.6171588376164436
Epoch 22/100, Loss: 1.6656839177012444
Epoch 23/100, Loss: 1.653546191751957
Epoch 24/100, Loss: 1.6632384955883026
Epoch 25/100, Loss: 1.6626197844743729
Epoch 26/100, Loss: 1.720491923391819
Epoch 27/100, Loss: 1.6637282818555832
Epoch 28/100, Loss: 1.5871332809329033
Epoch 29/100, Loss: 1.6709652990102768
Epoch 30/100, Loss: 1.6565144807100296
Epoch 31/100, Loss: 1.7219942286610603
Epoch 32/100, Loss: 1.688039056956768
Epoch 33/100, Loss: 1.6519825533032417


Epoch 34/100, Loss: 1.6566752567887306
Epoch 35/100, Loss: 1.708702053874731
Epoch 36/100, Loss: 1.718976803123951
Epoch 37/100, Loss: 1.6605894342064857
Epoch 38/100, Loss: 1.63645950704813
Epoch 39/100, Loss: 1.7132385596632957
Epoch 40/100, Loss: 1.6856501623988152
Epoch 41/100, Loss: 1.6613697856664658
Epoch 42/100, Loss: 1.6163360923528671
Epoch 43/100, Loss: 1.6704741194844246
Epoch 44/100, Loss: 1.6413547992706299
Epoch 45/100, Loss: 1.6098841801285744


Epoch 46/100, Loss: 1.5722157768905163
Epoch 47/100, Loss: 1.5966141819953918
Epoch 48/100, Loss: 1.6452529430389404
Epoch 49/100, Loss: 1.6848254650831223
Epoch 50/100, Loss: 1.7064484655857086
Epoch 51/100, Loss: 1.5910964608192444
Epoch 52/100, Loss: 1.6939346119761467
Epoch 53/100, Loss: 1.719371072947979
Epoch 54/100, Loss: 1.6264567375183105
Epoch 55/100, Loss: 1.6337088197469711
Epoch 56/100, Loss: 1.7079849019646645
Epoch 57/100, Loss: 1.6548756025731564
Epoch 58/100, Loss: 1.7335083931684494
Epoch 59/100, Loss: 1.6653303802013397
Epoch 60/100, Loss: 1.6426807418465614


Epoch 61/100, Loss: 1.856975071132183
Epoch 62/100, Loss: 1.72404994815588
Epoch 63/100, Loss: 1.6682732999324799
Epoch 64/100, Loss: 1.7441477328538895
Epoch 65/100, Loss: 1.60919239372015
Epoch 66/100, Loss: 1.6432043761014938
Epoch 67/100, Loss: 1.658076211810112
Epoch 68/100, Loss: 1.6577771976590157
Epoch 69/100, Loss: 1.6465725153684616
Epoch 70/100, Loss: 1.6426509767770767
Epoch 71/100, Loss: 1.6790865361690521
Epoch 72/100, Loss: 1.6045178323984146
Epoch 73/100, Loss: 1.7144941911101341
Epoch 74/100, Loss: 1.667393058538437
Epoch 75/100, Loss: 1.636790670454502


...

Epoch 26/100, Loss: 2.7686269506812096
Epoch 27/100, Loss: 2.531442940235138
Epoch 28/100, Loss: 2.6637111455202103
Epoch 29/100, Loss: 2.622016504406929
Epoch 30/100, Loss: 2.604912541806698
Epoch 31/100, Loss: 2.525516264140606
Epoch 32/100, Loss: 2.667979620397091
Epoch 33/100, Loss: 2.6598584353923798
Epoch 34/100, Loss: 2.4832020476460457
Epoch 35/100, Loss: 2.7336410582065582
Epoch 36/100, Loss: 2.644721694290638
Epoch 37/100, Loss: 2.756087303161621
Epoch 38/100, Loss: 2.5921481996774673


Epoch 39/100, Loss: 2.8406969383358955
Epoch 40/100, Loss: 2.8047315031290054
Epoch 41/100, Loss: 2.663146622478962
Epoch 42/100, Loss: 2.4388476088643074
Epoch 43/100, Loss: 2.548871234059334
Epoch 44/100, Loss: 2.712403267621994
Epoch 45/100, Loss: 2.7862106263637543
Epoch 46/100, Loss: 2.6620109006762505
Epoch 47/100, Loss: 3.4354204684495926
Epoch 48/100, Loss: 2.963871121406555
Epoch 49/100, Loss: 2.663363203406334
Epoch 50/100, Loss: 2.73380895704031
Epoch 51/100, Loss: 2.649555414915085
Epoch 52/100, Loss: 2.793110929429531
Epoch 53/100, Loss: 2.843827746808529
Epoch 54/100, Loss: 2.5095711052417755
Epoch 55/100, Loss: 2.5479510501027107


Epoch 56/100, Loss: 2.7781970277428627
Epoch 57/100, Loss: 2.6429812982678413
Epoch 58/100, Loss: 2.774746686220169
Epoch 59/100, Loss: 2.8008231222629547
Epoch 60/100, Loss: 2.982605718076229
Epoch 61/100, Loss: 2.3986879885196686
Epoch 62/100, Loss: 2.9599144235253334
Epoch 63/100, Loss: 2.5495382845401764
Epoch 64/100, Loss: 2.493255950510502
Epoch 65/100, Loss: 2.6697338819503784
Epoch 66/100, Loss: 2.8067704141139984
Epoch 67/100, Loss: 2.497072272002697
Epoch 68/100, Loss: 2.613462455570698
Epoch 69/100, Loss: 2.844176806509495
Epoch 70/100, Loss: 2.5992341935634613
Epoch 71/100, Loss: 2.6744366958737373


Epoch 72/100, Loss: 2.7849233373999596
Epoch 73/100, Loss: 2.656895585358143
Epoch 74/100, Loss: 2.586073860526085
Epoch 75/100, Loss: 2.5927793383598328
Epoch 76/100, Loss: 2.5704193860292435
Epoch 77/100, Loss: 2.393093816936016
Epoch 78/100, Loss: 2.556046210229397
Epoch 79/100, Loss: 2.6299502104520798
Epoch 80/100, Loss: 2.7576603144407272
Epoch 81/100, Loss: 2.8507591262459755
Epoch 82/100, Loss: 2.6723501086235046
Epoch 83/100, Loss: 2.485676921904087
Epoch 84/100, Loss: 2.627681501209736
Epoch 85/100, Loss: 2.936231069266796
Epoch 86/100, Loss: 2.795879326760769
Epoch 87/100, Loss: 2.753302216529846
Epoch 88/100, Loss: 2.434103786945343


Epoch 89/100, Loss: 2.5367953702807426
Epoch 90/100, Loss: 2.597625881433487
Epoch 91/100, Loss: 2.666776664555073
Epoch 92/100, Loss: 2.761579044163227
Epoch 93/100, Loss: 2.607075184583664
Epoch 94/100, Loss: 2.430142365396023
Epoch 95/100, Loss: 2.7342517226934433
Epoch 96/100, Loss: 2.926254153251648
Epoch 97/100, Loss: 2.6771984100341797
Epoch 98/100, Loss: 2.5935251340270042
Epoch 99/100, Loss: 2.8129759207367897
Epoch 100/100, Loss: 3.4032923206686974
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5861
